# Getting on the Machine · your first hour on Crux

This is Lab 00. By the end of it your notebook will submit a job to **Crux** (an ALCF PBS Pro cluster), that job will run on a real compute node, and its output will be back in your `~/lab00/` folder — with nothing typed into a terminal. Every later lab reuses this exact pattern, so it is worth the hour.

**You will:**
1. Prove you can `ssh` to Crux from this Hub, authenticating with your **MobilePASS+** token, and reach it without a fresh prompt for the rest of your session via ssh connection multiplexing.
2. Learn where things live at ALCF (home vs `/eagle` project vs `/local/scratch`) and why `filesystems=home:eagle` has to be on every job you submit.
3. Load a compiler with the `module` system and build a tiny C program on the login node.
4. Submit a **batch job** with `qsub`, watch it queue and run, and read its output back.
5. Do the same thing **interactively** with `qsub -I` — the debug workflow you will use all semester.

**Nothing here uses a GPU or MPI yet.** Those come in labs 05 (MPI) and 08 (GPU, on Polaris). Today is just "how do I get code onto Crux and get an answer back?"


## How this notebook works · Hub cells vs cluster cells

You are reading this notebook on a **Jupyter Hub that is not Crux**. There are three places code can run in this lab, and every cell below tells you which:

| Where | How it looks in the notebook | What it can do |
|---|---|---|
| **Hub** (this Jupyter kernel) | plain Python or `!command` | drive ssh/scp, run analysis, plot |
| **Crux login node** | `sshRun("...")` | edit files, compile, submit jobs |
| **Crux compute node** | inside a job script that `submitJob(...)` starts | run your program |

When you eventually work in the terminal (**File → New → Terminal** in JupyterLab), you can `source ~/lab00/labEnv.sh` and your shell will know the same host, user, and paths this notebook does.


In [ ]:
# [Hub] Load the shared lab toolkit (labHelpers.py ships in the course repo next to this notebook).
# It gives you preflight/checkpoint checks, sshRun/sshPut/sshGet, submitJob/waitJob, and
# the plotting primitives every later lab reuses.
from labHelpers import *


### Set up this lab's identity

`setupLab()` records which cluster you are targeting, which project (allocation) charges the job, which queue to submit to, and where your remote scratch lives. It exports those as environment variables that `sshRun`, `submitJob`, and every terminal shell you open will see.

The class project is **`UIC-CS455-Sp2027`** — already filled in below. **Only edit `HPC_USER`** to your ALCF username (e.g. `jsmith`).


In [ ]:
# [Hub] Change HPC_USER to your ALCF username, then run this cell.
env = setupLab(
    labName    = "lab00",
    host       = "crux",                                # ssh alias from your ~/.ssh/config
    remoteUser = os.environ.get("HPC_USER", "CHANGE_ME"),
    project    = "UIC-CS455-Sp2027",                    # the class allocation
    queue      = "debug",                               # small, fast-turnaround queue
    scratch    = f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}",
)


### Preflight · check your environment

These checks run before you touch the cluster. If any fail, the callout under it tells you exactly what to do. The most common failure is `ssh` — the whole of Part 1 walks you through fixing that.


In [ ]:
# [Hub] Environment health check.
preflight([
    check("ssh + scp on this Hub",
          lambda: (bool(shutil.which('ssh') and shutil.which('scp')),
                   f"ssh={shutil.which('ssh')}, scp={shutil.which('scp')}")),
    check("HPC_USER is set (not CHANGE_ME)",
          lambda: (os.environ.get('HPC_USER','CHANGE_ME') != 'CHANGE_ME',
                   os.environ.get('HPC_USER','(unset)')),
          hint="Edit the setupLab() cell above with your ALCF username and re-run it."),
    check("passwordless (multiplexed) ssh to Crux", sshReachable(),
          hint="See Part 1 below. Once your ssh control master is up, every ssh/scp to "
               "crux for the next 8 hours will reuse it - no MobilePASS+ re-prompt."),
    check("scheduler answers on Crux", schedulerAnswers(),
          hint="Once ssh works, this will pass. If it fails after ssh works, contact ALCF support."),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')), ('scratch', env.get('HPC_SCRATCH','?'))])


## Part 1 · ssh from the Hub to Crux, without retyping your token

ALCF authenticates with **MobilePASS+** — a one-time passcode from the app on your phone, not a fixed password. `ssh-copy-id` and simple public-key auth **do not work** at ALCF: every fresh ssh connection would ask for a new passcode, which would make this notebook unusable.

The solution is **ssh connection multiplexing**: you authenticate to Crux once (typing one MobilePASS+ passcode), and OpenSSH keeps that channel open in the background. Every later `ssh crux` or `scp ... crux:...` from this Hub — including every `sshRun` in this notebook — reuses the open channel, silently, for the next several hours.

> **🔀 On a Slurm system** (Crux is PBS Pro, but sites like UIC ACER Extreme run Slurm): > the ssh setup is identical. What differs is the scheduler you talk to *after* you are > in — labs use `submitJob()` to hide that difference.


**[Notebook cell]** Write an `~/.ssh/config` stanza that (a) points the alias `crux` at `crux.alcf.anl.gov` with your ALCF username, and (b) turns on connection multiplexing. The `ControlMaster auto` / `ControlPersist 8h` block is the whole trick.


In [ ]:
# [Hub] Append a Crux stanza to ~/.ssh/config (only if missing).
from pathlib import Path
sshDir    = Path.home() / ".ssh"
sshConfig = sshDir / "config"
sockDir   = sshDir / "cm"                # per-user socket dir for ssh multiplexing
sshDir.mkdir(mode=0o700, exist_ok=True)
sockDir.mkdir(mode=0o700, exist_ok=True)

stanza = f"""
# --- Crux (ALCF, PBS Pro) -----------------------------------------------
# MobilePASS+ prompts once; the control master keeps the channel open for 8h.
Host crux
    HostName crux.alcf.anl.gov
    User {os.environ['HPC_USER']}
    ControlMaster auto
    ControlPath   {sockDir}/%r@%h:%p
    ControlPersist 8h
    ServerAliveInterval 60
"""

existing = sshConfig.read_text() if sshConfig.exists() else ""
if "Host crux" not in existing:
    sshConfig.write_text(existing + stanza)
    sshConfig.chmod(0o600)
    print('added Crux stanza to ~/.ssh/config')
else:
    print('Crux stanza already present, leaving alone')
showFile(sshConfig, language='text')


**[Terminal]** Open **File → New → Terminal** in JupyterLab and log in ONCE to bring the multiplex channel up:

```bash
# From a Hub terminal, one time per ~8 hours of work:
ssh crux 'echo hello from crux'
#   -> ALCF prompts you for your MobilePASS+ passcode.
#      Type it. You should see:  hello from crux
#
# From this moment on, every ssh/scp/sshRun to crux reuses the open channel silently.
# Confirm it:
ssh crux hostname
#   -> no prompt, prints instantly.
```

When the second command returns instantly with no prompt, come back here and re-run the checkpoint below. If a lab in the middle of the semester says "ssh failing?" it usually means your 8-hour channel expired — just `ssh crux hostname` in a terminal to open a new one and keep going.


In [ ]:
checkpoint("Part 1 - ssh to Crux is multiplexed", [
    check("~/.ssh/config has a 'crux' host", fileContains("~/.ssh/config", "Host crux")),
    check("ControlMaster/ControlPersist configured",
          fileContains("~/.ssh/config", "ControlPersist"),
          hint="The stanza-writer cell above adds this; re-run it if you edited by hand."),
    check("ssh crux responds without a fresh MobilePASS+ prompt", sshReachable(),
          hint="Open a Hub terminal, run `ssh crux hostname`, enter your MobilePASS+ passcode "
               "ONCE. Then re-run this checkpoint."),
])


## Part 2 · Where things live at ALCF

Every HPC site has multiple filesystems, each tuned for a different purpose. Confusing them is the number-one source of "my job crashed but the code is fine" tickets. At ALCF:

| Path | Backing | Speed | Size | Use for |
|---|---|---|---|---|
| `~` (home) | NFS | slow | small (~50 GB) | source code, dotfiles, this repo |
| `/eagle/<project>/<user>/` | Lustre | fast, parallel | huge (TB) | job outputs, datasets, checkpoints |
| `/local/scratch/` (per-node) | node-local SSD | very fast | ~1 TB, per node | job-lifetime temp files, gone at exit |

**Rule you never break:** do not write big or hot data to `~`. It will slow the whole shared filesystem down for everyone at the facility. Every lab in this series stages its work under `$HPC_SCRATCH` (which `setupLab` set to `/eagle/UIC-CS455-Sp2027/<your-user>`).

### The `filesystems=` PBS directive · why every job needs it

At ALCF, every job **must declare which filesystems it will use**, up front, on the `#PBS` line. The scheduler uses this to hold a job back if the filesystem is under maintenance, so your job dies at submit time (nice) instead of at 4am mid-run (not nice).

You will see this on every job script in the course:

```bash
#PBS -l filesystems=home:eagle       # this job reads/writes home AND eagle
```

**If you forget it, your job will be rejected at submit.** Every job script in this repo has it.

> **🔀 On a Slurm system**: no equivalent — Slurm doesn't require declaring filesystems > up front. Just omit the line.


In [ ]:
# [Hub -> Crux login] Look around.
out, _ = sshRun("pwd && whoami && groups && df -h ~ /eagle 2>/dev/null | head -6")
print(out)


In [ ]:
# [Hub -> Crux login] Create your per-lab scratch directory (idempotent).
remoteLab = env['HPC_LAB_DIR']       # /eagle/UIC-CS455-Sp2027/<user>/lab00
out, _ = sshRun(f"mkdir -p {remoteLab} && ls -ld {remoteLab}")
print(out.strip().splitlines()[-1])


In [ ]:
checkpoint("Part 2 - filesystems and scratch", [
    check("HPC_SCRATCH is under /eagle",
          lambda: (env.get('HPC_SCRATCH','').startswith('/eagle'),
                   env.get('HPC_SCRATCH',''))),
    check("lab00 scratch dir exists on Crux", remoteFileExists(env['HPC_LAB_DIR'])),
])


## Part 3 · Modules · pick a compiler

Crux (like every HPC site) uses **environment modules** to switch between compilers, MPI implementations, and libraries. `module list` shows what is currently loaded; `module avail` lists what is installed; `module load <name>` puts one on your PATH.

For lab00 we just need a C compiler. Crux ships the HPE **Cray Programming Environment**, whose default C compiler wrapper is `cc` (it wraps GCC by default via `PrgEnv-gnu`).


In [ ]:
# [Hub -> Crux login] Poke the module system.
out, _ = sshRun("bash -lc 'module list 2>&1 && echo --- && module avail PrgEnv 2>&1 | head -20'")
print(out)


> **🔀 On a Slurm system**: `module` is not a scheduler thing — it comes from Lmod / > environment-modules and works identically everywhere. What differs is the *names* of > the modules (`gcc` vs `PrgEnv-gnu` vs `intel-oneapi`).


## Part 4 · Build a tiny C program on the login node

Two files, both written from the notebook, both `scp`'d to Crux. The whole point of this part is to teach the loop: **edit locally → scp → compile remotely → run**. Every later lab reuses it.


In [ ]:
# [Hub] Write hello.c locally.
helloSource = '''\
#include <stdio.h>
#include <unistd.h>
int main(void) {
    char host[256];
    gethostname(host, sizeof host);
    printf("hello from Crux compute node: %s\\n", host);
    return 0;
}
'''
local = Path(env['labDir']) / "hello.c"
local.write_text(helloSource)
showFile(local, language='c')


In [ ]:
# [Hub -> Crux] Copy it up, then compile on the login node.
sshPut(str(local), env['HPC_LAB_DIR'] + "/hello.c")
remoteLab = env['HPC_LAB_DIR']
out, _ = sshRun(f"bash -lc 'cd {remoteLab} && cc -O2 -Wall hello.c -o hello && ls -l hello'")
print(out)


In [ ]:
checkpoint("Part 4 - built hello on Crux", [
    check("hello.c present on Crux", remoteFileExists(env['HPC_LAB_DIR'] + "/hello.c")),
    check("hello binary present on Crux", remoteFileExists(env['HPC_LAB_DIR'] + "/hello")),
])


## Part 5 · Submit your first batch job

On the login node you compile and edit. You **do not run compute there** — it is shared with every other user at the facility. Real work goes through the scheduler.

A PBS Pro job script is a shell script with `#PBS` directives at the top that tell the scheduler what resources you want. Note the `filesystems=home:eagle` line — Part 2 explained why that has to be there.


In [ ]:
# [Hub] Write a PBS Pro job script that runs hello on a Crux compute node.
pbs = f'''\
#!/bin/bash
#PBS -N lab00Hello
#PBS -A {env['HPC_PROJECT']}
#PBS -q {env.get('HPC_QUEUE', 'debug')}
#PBS -l select=1:system=crux
#PBS -l walltime=00:05:00
#PBS -l filesystems=home:eagle
#PBS -j oe
#PBS -o {env['HPC_LAB_DIR']}/hello.out

cd {env['HPC_LAB_DIR']}
date
./hello
date
'''
jobScript = Path(env['labDir']) / "hello.pbs"
jobScript.write_text(pbs)
showFile(jobScript, language='bash')


> **🔀 On a Slurm system**, the same script becomes:
> ```bash
> #!/bin/bash
> #SBATCH --job-name=lab00Hello
> #SBATCH --account=UIC-CS455-Sp2027
> #SBATCH --partition=debug
> #SBATCH --nodes=1 --ntasks=1 --cpus-per-task=1
> #SBATCH --time=00:05:00
> #SBATCH --output=hello.out
> ```
> The `submitJob()` helper below picks the right submitter automatically (`qsub` vs > `sbatch`) based on what the remote cluster has, so this notebook cell works either way.


In [ ]:
# [Hub -> Crux] Ship the script up, submit it, and get a job id back.
sshPut(str(jobScript), env['HPC_LAB_DIR'] + "/hello.pbs")
jobId = submitJob(env['HPC_LAB_DIR'] + "/hello.pbs")
print("job id:", jobId)


In [ ]:
# [Hub] Watch it queue and run. The debug queue on Crux is usually a few minutes to start.
waitJob(jobId, pollSeconds=15, maxSeconds=1800)


In [ ]:
# [Hub -> Crux] Fetch the output back to the Hub and show it.
sshGet(env['HPC_LAB_DIR'] + "/hello.out", str(Path(env['labDir']) / "hello.out"))
showFile(Path(env['labDir']) / "hello.out", language='text', title='hello.out')


In [ ]:
checkpoint("Part 5 - first batch job", [
    check("hello.pbs present on Crux", remoteFileExists(env['HPC_LAB_DIR'] + "/hello.pbs")),
    check("hello.out was produced", fileExists(str(Path(env['labDir']) / "hello.out"))),
    check("hello.out contains a compute node hostname",
          fileContains(str(Path(env['labDir']) / "hello.out"), "hello from Crux compute node")),
])


## Part 6 · Interactive session · the debug workflow you will use all semester

Batch jobs are what you use once your code works. When it does *not* work, you want a shell prompt **on a compute node** so you can rebuild and rerun in seconds. That is `qsub -I`.

You do not run `qsub -I` from a notebook cell — it holds the terminal open. Instead, open **File → New → Terminal**, `source ~/lab00/labEnv.sh` (so `$HPC_LAB_DIR` and friends are set), then:

```bash
# In a Hub terminal:
source ~/lab00/labEnv.sh
ssh crux                                # reuses the multiplex channel, no prompt
# ... you are now on the Crux login node ...
qsub -I -A $HPC_PROJECT -q debug \\
        -l select=1:system=crux \\
        -l walltime=00:15:00 \\
        -l filesystems=home:eagle
# ... wait for the scheduler to give you a shell on a compute node ...
cd $HPC_LAB_DIR
./hello
hostname                                # note: you're on a compute node, not a login node
exit                                    # gives the allocation back
```

> **🔀 On a Slurm system**: `salloc --account=UIC-CS455-Sp2027 --partition=debug --nodes=1 --time=00:15:00`

You will lean on this in every lab from lab03 on: `qsub -I` to iterate, `qsub` (batch) to produce results.


In [ ]:
# [Hub] Show what's in your queue now (should be empty if the hello job already finished).
qstatTable()


## Wrap up

You now have every piece the rest of the course leans on: a multiplexed ssh channel, a per-lab scratch dir on Crux, a working compiler, a batch script pattern that includes the required `filesystems=` line, and a scheduler that answers you.

Every later lab starts the same way (`setupLab`, `preflight`, three checkpoints per part) and ends the same way (`labSummary`, `feedback`). Lab 08 onward switches `host='crux'` to `host='polaris'` in the `setupLab` call — because that is where the GPUs live — and everything else about this workflow keeps working unchanged.


### Lab scorecard


In [ ]:
labSummary("Getting on the Machine")


---
### One-minute feedback

What worked, what didn't, what should be clearer. This is anonymous to your classmates and goes straight to the instructor.


In [ ]:
feedback("Getting on the Machine")
